# 01 - Finetune Qwen2-VL (LoRA/QLoRA) trên Kaggle

**Mục tiêu:** finetune `Qwen/Qwen2-VL-2B-Instruct` cho bài toán *Vietnamese Textbook image captioning* (caption_detail).

**RUN_ID (định danh run)**
- `RUN_ID ∈ {A, B}` (notebook này **không** chạy baseline)
- `RUN_ID` sẽ được lưu vào `run_config.json` để notebook 02 tự nhận biết và đặt tên output theo tiền tố `{RUN_ID}_...`.

**Inputs**
- Hugging Face dataset: `bbdontcry/vietnamese-image-captioning` (các split `train/val/test`).
- Prompt (SYSTEM + PROMPT_DETAIL) được định nghĩa ngay trong notebook.

**Outputs** *(trong `OUTPUT_DIR=/kaggle/working/{RUN_ID}__Qwen2_VL_2B_Instruct/`)*  
- `adapter/` : LoRA adapter (weights)
- `processor/` : processor/tokenizer đã dùng
- `run_config.json` : metadata để notebook 02 dùng lại (run_id, model_id, min/max_pixels, prompt, generation config…)

**Notebook kế tiếp:** chạy `02-infer-qwen-vl.ipynb` (RUN_KIND=`adapter`) để sinh:
- `{RUN_ID}_predictions_test_detail.csv`
- `{RUN_ID}_inference_config.json`

> Tip: bật smoke test bằng biến môi trường `SMOKE_TEST=1` để chạy nhanh vài bước kiểm tra pipeline.

## 0) Chọn cấu hình chạy (A/B)

In [1]:
# =========================
# CONFIG PRESETS (A/B)
# =========================
RUN_ID = "A"  # "A" | "B"
DATASET_ID = "bbdontcry/vietnamese-image-captioning"

def pixels_from_visual_tokens(vt: int) -> int:
    # Qwen-VL hay quy đổi theo 28*28
    return vt * 28 * 28

PRESETS = {
    "A": dict(
        model_id="Qwen/Qwen2-VL-2B-Instruct",
        quantization=None,               
        min_pixels=pixels_from_visual_tokens(256),
        max_pixels=pixels_from_visual_tokens(1024),
        lora_r=16, lora_alpha=32, lora_dropout=0.05,
        lora_target="attn_only",          # "attn_only" | "attn_mlp"
        per_device_train_bs=1, per_device_eval_bs=1,
        grad_accum=16,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        weight_decay=0.0,
        lr_scheduler="linear",
        max_steps=800,
        eval_steps=100, save_steps=100, logging_steps=10,
        seed=42,
        gen_max_new_tokens_detail=2048,
    ),
    "B": dict(
        model_id="Qwen/Qwen2-VL-2B-Instruct",
        quantization=None,
        min_pixels=pixels_from_visual_tokens(256),
        max_pixels=pixels_from_visual_tokens(1280),
        lora_r=32, lora_alpha=64, lora_dropout=0.05,
        lora_target="attn_mlp",
        per_device_train_bs=1, per_device_eval_bs=1,
        grad_accum=16,
        learning_rate=1e-4,
        warmup_ratio=0.03,
        weight_decay=0.0,
        lr_scheduler="linear",
        max_steps=800,
        eval_steps=100, save_steps=100, logging_steps=10,
        seed=42,
        gen_max_new_tokens_detail=2048,
    ),
}
cfg = PRESETS[RUN_ID].copy()

SMOKE_TEST = "1" # "1" để chạy smoke test
if SMOKE_TEST:
    # quick sanity run
    cfg["max_steps"] = 10
    cfg["eval_steps"] = 5
    cfg["save_steps"] = 5
    cfg["logging_steps"] = 1

cfg

{'model_id': 'Qwen/Qwen2-VL-2B-Instruct',
 'quantization': None,
 'min_pixels': 200704,
 'max_pixels': 802816,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'lora_target': 'attn_only',
 'per_device_train_bs': 1,
 'per_device_eval_bs': 1,
 'grad_accum': 16,
 'learning_rate': 0.0002,
 'warmup_ratio': 0.03,
 'weight_decay': 0.0,
 'lr_scheduler': 'linear',
 'max_steps': 10,
 'eval_steps': 5,
 'save_steps': 5,
 'logging_steps': 1,
 'seed': 42,
 'gen_max_new_tokens_detail': 2048}

## 1) Paths

In [2]:
from pathlib import Path

OUTPUT_ROOT = Path("/kaggle/working")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = f"{RUN_ID}__" + cfg["model_id"].split("/")[-1].replace(".", "_").replace("-", "_")
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR =", OUTPUT_DIR)

OUTPUT_DIR = /kaggle/working/A__Qwen2_VL_2B_Instruct


## 2) Install deps

In [3]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q"] + pkgs)

pip_install([
    "datasets>=2.18.0",
    "accelerate>=0.33.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "evaluate>=0.4.0",
    "sacrebleu>=2.4.0",
    "rouge_score>=0.1.2",
])

# TRL (git) để có VLM collation ổn định
pip_install(["git+https://github.com/huggingface/trl.git"])

# Transformers: Qwen2.5-VL đôi khi cần bản rất mới
need_source = "Qwen2.5" in cfg["model_id"]
if need_source:
    pip_install(["git+https://github.com/huggingface/transformers", "accelerate"])
else:
    pip_install(["transformers>=4.44.0"])

print("Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.6 MB/s eta 0:00:00
Done.


## 3) Imports + GPU

In [4]:
import random
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def pick_amp_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, minor = torch.cuda.get_device_capability()
    return torch.bfloat16 if major >= 8 else torch.float16

AMP_DTYPE = pick_amp_dtype()

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| cap:", torch.cuda.get_device_capability(0))
print("AMP_DTYPE:", AMP_DTYPE)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg["seed"])

CUDA: True
GPU: Tesla T4 | cap: (7, 5)
AMP_DTYPE: torch.float16


## 4) Load dataset + detect columns

In [5]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID)

def get_split(ds_dict, candidates):
    for k in candidates:
        if k in ds_dict:
            return ds_dict[k], k
    raise KeyError(f"Missing split. Available={list(ds_dict.keys())}")

train_raw, _ = get_split(raw, ["train"])
val_raw, _   = get_split(raw, ["val","validation","dev"])
test_raw, _  = get_split(raw, ["test"])

print("Lens:", len(train_raw), len(val_raw), len(test_raw))
print("Cols:", train_raw.column_names)

def pick_first(cols, candidates):
    for c in candidates:
        if c in cols: return c
    return None

img_col = pick_first(train_raw.column_names, ["image","img","image_pil"])
cap_detail_col = pick_first(train_raw.column_names, ["caption_detail","detail","caption_long","caption"])

print("img_col:", img_col)
print("cap_detail_col:", cap_detail_col)

assert img_col is not None
assert cap_detail_col is not None, "Dataset phải có cột caption_detail (hoặc alias)."

README.md:   0%|          | 0.00/927 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/330M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/345M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/156M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/981 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/123 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/125 [00:00<?, ? examples/s]

Lens: 981 123 125
Cols: ['id', 'image', 'caption_short', 'caption_detail', 'metadata_type', 'metadata_collection', 'metadata_title', 'metadata_grade', 'metadata_subject', 'metadata_author', 'metadata_publisher']
img_col: image
cap_detail_col: caption_detail


## 5) Prompt (detail only)

In [6]:
SYSTEM_MESSAGE = (
    "Bạn là một Vision-Language Model hỗ trợ người khiếm thị bằng cách tạo mô tả ảnh bằng tiếng Việt cho trang sách giáo khoa. "
    "Chỉ mô tả những gì thấy trong ảnh, không suy đoán."
    "Khi OCR, trích toàn bộ chữ nhìn thấy, theo thứ tự trên xuống dưới, trái sang phải"
)

PROMPT_DETAIL = (
    "Hãy thuyết minh chi tiết trang SGK trong ảnh theo luồng đọc từ trên xuống dưới. "
    "Khi gặp chữ trong ảnh, hãy đưa vào đúng ngữ cảnh và trích nguyên văn trong ngoặc kép. "
    "Không suy luận."
)

## 6) Format dataset (caption_detail only)

In [7]:
from datasets import concatenate_datasets
from datasets.utils.logging import set_verbosity_info
set_verbosity_info()

TRAIN_LIMIT = 32 if "SMOKE_TEST" in globals() and SMOKE_TEST else None
VAL_LIMIT   = 8  if "SMOKE_TEST" in globals() and SMOKE_TEST else None

train_ds = train_raw if TRAIN_LIMIT is None else train_raw.select(range(min(TRAIN_LIMIT, len(train_raw))))
val_ds   = val_raw   if VAL_LIMIT   is None else val_raw.select(range(min(VAL_LIMIT, len(val_raw))))

# content đều là STRING để Arrow không bị "mix list/non-list"
# user chứa đúng 1 <|image_pad|> tương ứng 1 ảnh
VISION_PREFIX = "<|vision_start|><|image_pad|><|vision_end|>\n"

def build_messages(answer_text):
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": VISION_PREFIX + PROMPT_DETAIL},
        {"role": "assistant", "content": str(answer_text)},
    ]

def format_detail(ds, batch_size=32, desc="format_detail"):
    ds2 = ds.filter(
        lambda ex: ex[cap_detail_col] is not None and str(ex[cap_detail_col]).strip() != "",
        desc="filter_detail"
    )

    def _map(batch):
        imgs = batch[img_col]
        caps = batch[cap_detail_col]
        msgs = [build_messages(cap) for cap in caps]
        tasks = ["detail"] * len(caps)
        return {"image": imgs, "messages": msgs, "task": tasks}

    return ds2.map(
        _map,
        batched=True,
        batch_size=batch_size,
        remove_columns=ds2.column_names,
        desc=desc,
        load_from_cache_file=False,
    )

train_fmt = format_detail(train_ds, batch_size=32, desc="format_train_detail")
val_fmt   = format_detail(val_ds, batch_size=32, desc="format_val_detail")

print("train_fmt:", len(train_fmt), "val_fmt:", len(val_fmt))
print(train_fmt[0]["task"])
print(train_fmt.column_names)

filter_detail:   0%|          | 0/32 [00:00<?, ? examples/s]

Caching processed dataset at /root/.cache/huggingface/datasets/bbdontcry___vietnamese-image-captioning/default/0.0.0/21b12179886da25c6118364bb8084db9ab521ad6/cache-6918f522c06376b7.arrow


format_train_detail:   0%|          | 0/32 [00:00<?, ? examples/s]

Caching processed dataset at /root/.cache/huggingface/datasets/bbdontcry___vietnamese-image-captioning/default/0.0.0/21b12179886da25c6118364bb8084db9ab521ad6/cache-49edf2b14be0d9b0.arrow


filter_detail:   0%|          | 0/8 [00:00<?, ? examples/s]

Caching processed dataset at /root/.cache/huggingface/datasets/bbdontcry___vietnamese-image-captioning/default/0.0.0/21b12179886da25c6118364bb8084db9ab521ad6/cache-ba50fa08e4d3ec59.arrow


format_val_detail:   0%|          | 0/8 [00:00<?, ? examples/s]

Caching processed dataset at /root/.cache/huggingface/datasets/bbdontcry___vietnamese-image-captioning/default/0.0.0/21b12179886da25c6118364bb8084db9ab521ad6/cache-290f00590d4e944d.arrow


train_fmt: 32 val_fmt: 8
detail
['image', 'messages', 'task']


## 7) Load processor + model

In [8]:
from transformers import AutoProcessor, BitsAndBytesConfig

processor = AutoProcessor.from_pretrained(
    cfg["model_id"],
    min_pixels=cfg["min_pixels"],
    max_pixels=cfg["max_pixels"],
)

# Common safe default
if hasattr(processor, "tokenizer") and hasattr(processor.tokenizer, "padding_side"):
    processor.tokenizer.padding_side = "right"

use_4bit = (cfg["quantization"] == "4bit")
bnb = None
if use_4bit:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=AMP_DTYPE,
    )

model_id = cfg["model_id"]
if "Qwen2.5" in model_id:
    from transformers import Qwen2_5_VLForConditionalGeneration as ModelClass
else:
    from transformers import Qwen2VLForConditionalGeneration as ModelClass

model = ModelClass.from_pretrained(
    model_id,
    torch_dtype=AMP_DTYPE if not use_4bit else None,
    quantization_config=bnb,
).to("cuda")

print("Loaded:", model_id, "| 4bit:", use_4bit)

2026-01-04 08:07:21.904547: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767514042.133162      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767514042.204284      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767514042.759703      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767514042.759747      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767514042.759750      55 computation_placer.cc:177] computation placer alr

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2-VL-2B-Instruct | 4bit: False


## 8) Train (TRL SFTTrainer + LoRA)

In [9]:
from dataclasses import dataclass

def _find_sublist(haystack, needle):
    n = len(needle)
    for i in range(len(haystack) - n + 1):
        if haystack[i:i+n] == needle:
            return i
    return None

ASSISTANT_PREFIX = "<|im_start|>assistant"
assistant_ids = processor.tokenizer.encode(ASSISTANT_PREFIX, add_special_tokens=False)

@dataclass
class QwenVLCollator:
    processor: any
    def __call__(self, examples):
        texts = [self.processor.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
                 for ex in examples]
        images = [ex["image"] for ex in examples]  # 1 ảnh / sample

        batch = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        labels = batch["input_ids"].clone()
        for i in range(labels.size(0)):
            ids = batch["input_ids"][i].tolist()
            pos = _find_sublist(ids, assistant_ids)
            if pos is None:
                labels[i, :] = -100
                continue
            start = pos + len(assistant_ids)
            labels[i, :start] = -100

        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch

data_collator = QwenVLCollator(processor)

In [10]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def target_modules(kind: str):
    attn = ["q_proj","k_proj","v_proj","o_proj"]
    mlp  = ["gate_proj","up_proj","down_proj"]
    return attn if kind == "attn_only" else (attn + mlp)

peft_config = LoraConfig(
    r=cfg["lora_r"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=cfg["lora_dropout"],
    bias="none",
    target_modules=target_modules(cfg["lora_target"]),
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_length=None,
    dataset_kwargs={"skip_prepare_dataset": True},

    per_device_train_batch_size=cfg["per_device_train_bs"],
    per_device_eval_batch_size=cfg["per_device_eval_bs"],
    gradient_accumulation_steps=cfg["grad_accum"],

    max_steps=cfg["max_steps"],
    learning_rate=cfg["learning_rate"],
    warmup_ratio=cfg["warmup_ratio"],
    weight_decay=cfg["weight_decay"],
    lr_scheduler_type=cfg["lr_scheduler"],

    eval_strategy="steps",
    eval_steps=cfg["eval_steps"],
    save_strategy="steps",
    save_steps=cfg["save_steps"],
    save_total_limit=2,

    logging_steps=cfg["logging_steps"],
    report_to="none",

    fp16=(AMP_DTYPE == torch.float16),
    bf16=(AMP_DTYPE == torch.bfloat16),

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_fmt,
    eval_dataset=val_fmt,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Step,Training Loss,Validation Loss
5,1.488200,1.562044
10,1.398200,1.508052


TrainOutput(global_step=10, training_loss=1.486447846889496, metrics={'train_runtime': 353.2586, 'train_samples_per_second': 0.453, 'train_steps_per_second': 0.028, 'total_flos': 2571624697743360.0, 'train_loss': 1.486447846889496})

## 9) Save adapter + run_config.json

In [11]:
import json, time

adapter_dir = Path(OUTPUT_DIR) / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(adapter_dir)
processor.save_pretrained(Path(OUTPUT_DIR) / "processor")

run_config = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_id": RUN_ID,
    "preset": RUN_ID,
    "dataset_id": DATASET_ID,
    "model_id": cfg["model_id"],
    "quantization": cfg["quantization"],
    "min_pixels": cfg["min_pixels"],
    "max_pixels": cfg["max_pixels"],
    "system_message": SYSTEM_MESSAGE,
    "prompt_detail": PROMPT_DETAIL,
    "gen": {
        "max_new_tokens_detail": cfg["gen_max_new_tokens_detail"],
        "temperature": 0.2,
        "top_p": 0.9,
        "do_sample": False,
    },
}

with open(Path(OUTPUT_DIR) / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

print("Saved:", adapter_dir)

Saved: /kaggle/working/A__Qwen2_VL_2B_Instruct/adapter
